# VisionScale AI - Kaggle T4 GPU Notebook

This notebook is designed for Kaggle with a free NVIDIA T4 GPU. It installs the project, verifies the environment, and runs the project setup and sample pipeline.

Steps before running:
- In Kaggle Notebook settings, set `Accelerator` to `GPU`
- Choose `T4 x 1` if available
- Keep the notebook in Python mode


In [ ]:
!nvidia-smi
!python --version
!pip --version

## 1) Clone the repository

Run the cell below to pull the project into your Kaggle notebook working directory.

In [ ]:
!git clone https://github.com/webdevaditya1/VisionScale-AI.git
%cd VisionScale-AI
!ls -la

## 2) Install FFmpeg and project dependencies

FFmpeg is required for video processing. Use Kaggle's existing Python environment; no venv activation is needed.

In [ ]:
!apt-get update
!apt-get install -y ffmpeg
!ffmpeg -version
!python -m pip install -r requirements-gpu.txt
!python -m pip check

## 3) Run verification tests

This confirms that the project is working correctly on the Kaggle GPU environment.

In [ ]:
!python -m pytest tests/ -v

## 4) Check the detected GPU and hardware profile

This should report a CUDA-enabled T4 device and confirm FP16 is available.

In [ ]:
from visionscale.core.hardware import detect_hardware
print(detect_hardware().summary())

## 5) Upload or locate your input video

Use the Kaggle file upload panel or a path from `/kaggle/input` or `/kaggle/working`.

In [ ]:
import glob
video_files = []
for base in ['/kaggle/input', '/kaggle/working']:
    video_files.extend(glob.glob(base + '/**/*.*', recursive=True))

video_files = sorted({p for p in video_files if p.lower().endswith(('.mp4', '.mov', '.mkv', '.avi', '.webm', '.m4v'))})
print('Found videos:')
for v in video_files:
    print(v)

if not video_files:
    print('No video files found yet. Upload one from the Kaggle panel and rerun this cell.')
else:
    print('Using first video:')
    print(video_files[0])
    VIDEO_PATH = video_files[0]

## 6) Enhance the video and export an MP4

This cell runs tiled Real-ESRGAN inference frame by frame, preserves the source audio when possible, and writes the result under `/kaggle/working/visionscale_work/outputs`. The first run downloads the model weights, so enable Internet in Kaggle notebook settings. Set the target to `P4K` for a 4K output or change it to `P1080`.

In [ ]:
from visionscale.config import VisionScaleConfig, ResolutionTarget, ContentCategory
from visionscale.pipeline import enhance_video

from pathlib import Path
VIDEO_PATH = globals().get('VIDEO_PATH', '/kaggle/working/input_video.mp4')
if not Path(VIDEO_PATH).is_file():
    raise FileNotFoundError(f'Video not found: {VIDEO_PATH}. Set VIDEO_PATH to a file under /kaggle/input or /kaggle/working.')

config = VisionScaleConfig()
config.processing.target_resolution = ResolutionTarget.P4K
config.processing.content_category = ContentCategory.AUTO
config.processing.use_fp16 = True

def show_progress(done, total):
    if done == total or done % 25 == 0:
        print(f'Processed {done}/{total} frames')

output_path = enhance_video(VIDEO_PATH, config, progress_callback=show_progress)
print(f'Enhanced video saved to: {output_path}')

## 7) Find and download the result

Open Kaggle's output/file browser and download the MP4 at the path printed above. For long videos, processing is frame-by-frame and can take a while; start with a short clip first.

## 8) Optional: clean GPU memory

Use this if you are restarting the run or hit memory issues.

In [ ]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
print('GPU memory cache cleared.')